# 05 · Export preview

The exported `.avatar` file, posed and rendered the way a game runtime would: through `AvatarRuntime`, which reads only the file. Run `avatar-splats export` first.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import roma
import torch

from avatar_splats.cameras import Camera
from avatar_splats.config import load_config
from avatar_splats.export import AvatarRuntime, bind_relative, exported_path, read_avatar
from avatar_splats.gaussians import render

config = load_config("../configs/default.yaml")
device = config.training.device
file = read_avatar(exported_path(config))
runtime = AvatarRuntime(file).to(device)
print({key: value for key, value in file.metadata.items() if key not in ("bones", "expressions")})
print(len(runtime.gaussians), "Gaussians,", len(runtime.bone_names), "bones")

A pose is a rotation per bone. Here they are given in world axes (x to the avatar's left, y to its back, z up) and converted to the bind-relative rotations the file's skeleton uses.

In [ ]:
def pose(**world_rotvecs):
    """Bind-relative rotations from rotation vectors in world axes, by bone name."""
    rotations = torch.eye(3, device=device).repeat(1, len(runtime.bone_names), 1, 1)
    for bone, rotvec in world_rotvecs.items():
        index = runtime.bone_names.index(bone.replace("_", "."))
        rotations[0, index] = roma.rotvec_to_rotmat(
            torch.tensor(rotvec, device=device, dtype=torch.float32)
        )
    return bind_relative(runtime.rig.bind_poses, rotations)


def expressions(**weights):
    values = torch.zeros(1, len(runtime.expression_names), device=device)
    for name, weight in weights.items():
        values[0, runtime.expression_names.index(name)] = weight
    return values


def turntable(
    rotations, expression_weights=None, angles=(0, 45, 90, 180, 270), size=(480, 800), crop=None
):
    """Render the posed avatar from several sides; ``crop`` frames only what is above it."""
    vertices = runtime.pose(rotations, expression_weights)[0]
    framed = vertices if crop is None else vertices[vertices[:, 2] > crop]
    low, high = framed.min(dim=0).values, framed.max(dim=0).values
    camera = Camera.fixed(*size, fov_y_degrees=30).to(device)
    distance = camera.depth_for_height(1.1 * float(high[2] - low[2]), size[1])
    images = []
    for angle in angles:
        turn = roma.rotvec_to_rotmat(torch.tensor([0, 0, math.radians(angle)], device=device))
        placed = (vertices - (low + high) / 2) @ turn.T + torch.tensor(
            [0, distance, 0], device=device
        )
        with torch.no_grad():
            means, quats, scales, opacities, sh = runtime.place(placed)
            white = torch.ones(3, device=device)
            image, _, _ = render(
                means, quats, scales, opacities, sh, camera, runtime.sh_degree, white
            )
        images.append(image.cpu().numpy())
    plt.figure(figsize=(4 * len(angles), 7))
    plt.imshow(np.concatenate(images, axis=1))
    plt.axis("off")

## Rest pose

In [ ]:
turntable(pose())

## A new pose

Arms raised, a step forward, the head turned: poses the capture never showed.

In [ ]:
turntable(
    pose(
        upperarm01_L=[0, -1.2, 0],
        upperarm01_R=[0, 1.2, 0],
        upperleg01_L=[0.5, 0, 0],
        lowerleg01_L=[-0.6, 0, 0],
        head=[0, 0, 0.5],
    )
)

## Expressions

ARKit-named blendshapes on the head, here at the front view only.

In [ ]:
head_top = float(runtime.rig.rest_vertices[:, 2].max())
for weights in (
    {},
    {"jawOpen": 0.7},
    {"mouthSmile_L": 1, "mouthSmile_R": 1},
    {"eyeBlink_L": 1, "eyeBlink_R": 1},
):
    turntable(pose(), expressions(**weights), angles=(0,), size=(480, 480), crop=head_top - 0.3)
    plt.title(", ".join(weights) or "neutral")